# CS 521 HW4, Problem 2: Certified Training for Multiple Norms (CURE)

Open this notebook in Google Colab and select a GPU runtime (*Runtime > Change runtime type > T4 GPU*).

Complete each part that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the verifier, the evaluation or the default hyperparameters.
Change them only if the homework tells you to.

**Threat model.** Let $f$ be the network and $x$ an image with the label $y$. The network is *certified* at $x$
for the norm $p$ if the verifier proves this: $f_y(x') - f_j(x') > 0$ for all $j \neq y$ and all $x'$ with
$\|x' - x\|_p \leq \epsilon_p$. For $\ell_\infty$, $x'$ must also be in $[0, 1]^{784}$.
We use $\epsilon_\infty = 0.1$ and $\epsilon_2 = 1.0$.
The *union certified accuracy* is the percentage of test images where the network is certified for the two norms.

In [1]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

EPS = {"linf": 0.1, "l2": 1.0}     # the radii of the threat model
NORMS = ["linf", "l2"]

device: cuda


# Data and model

The data and the network are the same as in Problem 1. The network has 3 hidden layers with 50 neurons.
It has no normalization layer.

In [2]:
def load_mnist(train):
    ds = torchvision.datasets.MNIST("data", train=train, download=True)
    x = ds.data.float().div(255).unsqueeze(1)  # (n, 1, 28, 28)
    return x.to(device), ds.targets.to(device)


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for i in range(0, len(x), batch_size):
        j = idx[i:i + batch_size]
        yield x[j], y[j]


x_train, y_train = load_mnist(train=True)
x_test, y_test = load_mnist(train=False)
print(x_train.shape, x_test.shape)


def make_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 10),
    ).to(device)


def linear_layers(model):
    return [m for m in model if isinstance(m, nn.Linear)]

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.08MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 135kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.29MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.51MB/s]


torch.Size([60000, 1, 28, 28]) torch.Size([10000, 1, 28, 28])


# Attacks

`pgd(model, x, y, eps, norm, steps)` is an untargeted PGD attack with a random start.
Use `norm="linf"` or `norm="l2"`. The function returns the adversarial images of the restart with the highest loss.

In [3]:
def _project(delta, eps, norm):
    if norm == "linf":
        return delta.clamp(-eps, eps)
    n = delta.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
    return delta * (eps / n).clamp(max=1.0)


def pgd(model, x, y, eps, norm, steps=10, alpha=None, restarts=1):
    alpha = alpha or 2.5 * eps / steps
    best_x, best_loss = x.clone(), torch.full((len(x),), -float("inf"), device=x.device)
    for _ in range(restarts):
        if norm == "linf":
            delta = (torch.rand_like(x) * 2 - 1) * eps
        else:
            delta = torch.randn_like(x)
            r = torch.rand(len(x), 1, 1, 1, device=x.device) * eps
            delta = delta / delta.flatten(1).norm(dim=1).view(-1, 1, 1, 1) * r
        delta = (x + delta).clamp(0, 1) - x
        for _ in range(steps):
            delta.requires_grad_(True)
            loss = F.cross_entropy(model(x + delta), y)
            grad, = torch.autograd.grad(loss, delta)
            with torch.no_grad():
                if norm == "linf":
                    step = grad.sign()
                else:
                    step = grad / grad.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
                delta = _project(delta + alpha * step, eps, norm)
                delta = (x + delta).clamp(0, 1) - x
        with torch.no_grad():
            loss = F.cross_entropy(model(x + delta), y, reduction="none")
            better = loss > best_loss
            best_x[better], best_loss[better] = (x + delta)[better], loss[better]
    return best_x.detach()

# Verifier

A *region* has a center `c` (a batch of images), a radius `r` and a norm.
For $\ell_\infty$, the region is the box $[c - r, c + r] \cap [0, 1]^{784}$.
For $\ell_2$, the region is the ball $\{x' : \|x' - c\|_2 \leq r\}$. We do not intersect the ball with $[0, 1]^{784}$.
This is sound, but the bounds are a little looser.

* `ibp_margin_bounds(model, c, r, norm, y)` is IBP from Problem 1. For $\ell_2$, the bounds of the first linear layer
  are exact: $W_i c + b_i \pm r \|W_i\|_2$ for each neuron $i$. The last linear layer is merged with the specification.
* `crown_margin_bounds(model, c, r, norm, y)` is CROWN. At the input, $\min_{x'} a^\top x'$ over the $\ell_2$ ball is
  $a^\top c - r \|a\|_2$.
* `certified(model, x, y, norm)` uses the larger of the two lower bounds.

The two bound functions return a tensor `m` with the shape `(batch, 10)`.
The value `m[:, j]` is a lower bound of $f_y - f_j$ on the region, and `m[:, y] = 0`. Do not change the verifier.

In [4]:
def first_layer_bounds(layer, c, r, norm):
    c = c.flatten(1)
    if norm == "linf":
        l, u = (c - r).clamp(0, 1), (c + r).clamp(0, 1)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
    else:
        mid, rad = layer(c), r * layer.weight.norm(dim=1)
    return mid - rad, mid + rad


def spec_matrix(y, n_classes=10):
    eye = torch.eye(n_classes, device=y.device)
    return eye[y].unsqueeze(1) - eye.unsqueeze(0)  # (batch, 10, 10): row j is e_y - e_j


def ibp_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    l, u = first_layer_bounds(layers[0], c, r, norm)
    for layer in layers[1:-1]:
        l, u = F.relu(l), F.relu(u)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
        l, u = mid - rad, mid + rad
    l, u = F.relu(l), F.relu(u)
    last = layers[-1]
    w = last.weight[y].unsqueeze(1) - last.weight.unsqueeze(0)  # (batch, 10, n): W_y - W_j
    b = last.bias[y].unsqueeze(1) - last.bias.unsqueeze(0)
    mid, rad = (u + l) / 2, (u - l) / 2
    return (w @ mid.unsqueeze(2)).squeeze(2) - (w.abs() @ rad.unsqueeze(2)).squeeze(2) + b


def _relu_relaxation(l, u):
    active, unstable = l >= 0, (l < 0) & (u > 0)
    s = torch.where(active, torch.ones_like(l), torch.zeros_like(l))
    s = torch.where(unstable, u / (u - l).clamp(min=1e-12), s)
    c = torch.where(unstable, -s * l, torch.zeros_like(l))
    alpha = torch.where(active | (unstable & (u > -l)), torch.ones_like(l), torch.zeros_like(l))
    return alpha, s, c


def _backward_lower(A, layers, bounds, c0, r, norm):
    # Lower bound of A z, where z is the output of layers[-1] and bounds[i] are the bounds of the output of layers[i].
    const = torch.zeros(A.shape[:2], device=A.device)
    for i in range(len(layers) - 1, -1, -1):
        const = const + A @ layers[i].bias
        A = A @ layers[i].weight
        if i > 0:
            alpha, s, c = _relu_relaxation(*bounds[i - 1])
            a_pos, a_neg = A.clamp(min=0), A.clamp(max=0)
            const = const + (a_neg * c.unsqueeze(1)).sum(2)
            A = a_pos * alpha.unsqueeze(1) + a_neg * s.unsqueeze(1)
    c0 = c0.flatten(1)
    if norm == "linf":
        l0, u0 = (c0 - r).clamp(0, 1), (c0 + r).clamp(0, 1)
        return const + (A.clamp(min=0) * l0.unsqueeze(1)).sum(2) + (A.clamp(max=0) * u0.unsqueeze(1)).sum(2)
    return const + (A @ c0.unsqueeze(2)).squeeze(2) - r * A.norm(dim=2)


def crown_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    bounds = []
    for k in range(len(layers) - 1):
        eye = torch.eye(layers[k].out_features, device=c.device).expand(len(c), -1, -1)
        lo = _backward_lower(eye, layers[:k + 1], bounds, c, r, norm)
        hi = -_backward_lower(-eye, layers[:k + 1], bounds, c, r, norm)
        bounds.append((lo, hi))
    return _backward_lower(spec_matrix(y), layers, bounds, c, r, norm)


@torch.no_grad()
def certified(model, x, y, norm, eps=None):
    eps = EPS[norm] if eps is None else eps
    m = torch.maximum(ibp_margin_bounds(model, x, eps, norm, y), crown_margin_bounds(model, x, eps, norm, y))
    m[torch.arange(len(y)), y] = float("inf")
    return m.min(dim=1).values > 0

# Evaluation

`evaluate(model, name)` uses the 10,000 test images. It returns the clean accuracy, the certified accuracy for each
norm and the union certified accuracy. With `attacks=True`, it also returns the PGD accuracy (50 steps) for each norm
and for the union. The PGD accuracy is an upper bound of the true robust accuracy.
Use `evaluate` for all the numbers in your report.

In [5]:
def evaluate(model, name, attacks=False, n=10000, batch_size=1000):
    model.eval()
    counts = {k: 0 for k in ["clean", "linf", "l2", "union", "pgd_linf", "pgd_l2", "pgd_union"]}
    for x, y in batches(x_test[:n], y_test[:n], batch_size, False):
        with torch.no_grad():
            correct = model(x).argmax(1) == y
        cert = {p: certified(model, x, y, p) & correct for p in NORMS}
        counts["clean"] += correct.sum().item()
        counts["linf"] += cert["linf"].sum().item()
        counts["l2"] += cert["l2"].sum().item()
        counts["union"] += (cert["linf"] & cert["l2"]).sum().item()
        if attacks:
            rob = {}
            for p in NORMS:
                x_adv = pgd(model, x, y, EPS[p], p, steps=50)
                with torch.no_grad():
                    rob[p] = correct & (model(x_adv).argmax(1) == y)
                counts["pgd_" + p] += rob[p].sum().item()
            counts["pgd_union"] += (rob["linf"] & rob["l2"]).sum().item()
    res = {"name": name}
    res.update({k: 100 * v / n for k, v in counts.items() if attacks or not k.startswith("pgd")})
    return res


def print_results(rows):
    for r in rows:
        print(f"{r['name']:<26}" + "  ".join(f"{k} {v:6.2f}" for k, v in r.items()
                                     if k != "time" and isinstance(v, float)) +
              (f"  time/epoch {r['time']:5.1f} s" if "time" in r else ""))

# Training loop

For each batch, `train(model, loss_fn)` calls `loss_fn(model, x, y, t)` and does one optimizer step.
The value `t` is the training progress in epochs (a float). The schedule is the same as in Problem 1:

* For $t < 1$, `eps_scale(t)` is 0 (standard training).
* For $1 \leq t < 11$, `eps_scale(t)` increases linearly from 0 to 1.
* For $t \geq 11$, `eps_scale(t)` is 1.

A loss function uses the radius `eps_scale(t) * EPS[norm]`.

If you give the argument `gp_beta`, `train` uses `gradient_projection` (part (b)) when the radius is at its
final value ($t \geq 11$).

In [6]:
WARMUP, RAMP = 1, 10
LR = 4e-3


def eps_scale(t):
    return 0.0 if t <= WARMUP else min((t - WARMUP) / RAMP, 1.0)


def train(model, loss_fn, epochs=20, lr=LR, batch_size=512, seed=0, gp_beta=None, verbose=True):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[15], gamma=0.1)
    params = [p for p in model.parameters() if p.requires_grad]
    n_batches = (len(x_train) + batch_size - 1) // batch_size
    start = time.time()
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for i, (x, y) in enumerate(batches(x_train, y_train, batch_size, True)):
            t = epoch + i / n_batches
            loss = loss_fn(model, x, y, t)
            opt.zero_grad()
            if gp_beta is not None and t >= WARMUP + RAMP:
                g_c = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, gn, gc in zip(params, g_n, g_c):
                    p.grad = gradient_projection(gn, gc, gp_beta)
            else:
                loss.backward()
            opt.step()
            total += loss.item()
        sched.step()
        if verbose:
            print(f"epoch {epoch + 1:2d}  loss {total / n_batches:.4f}  time {time.time() - start:.0f} s")
    return (time.time() - start) / epochs


def standard_loss(model, x, y, t):
    return F.cross_entropy(model(x), y)

# (a) Single-norm certified training

CURE uses the SABR loss from the lectures for each norm. SABR finds an adversarial example $x^*$ with PGD, and then it
calculates the IBP loss on a small ball of radius $\tau = \lambda \epsilon$ around $x^*$. PGD searches for $x^*$ in the
ball of radius $\epsilon - \tau$ around $x$, so that the small ball is inside the full ball.

1. `propagation_region`: Find a center $x^*$ with `pgd` (5 steps) in the ball of radius $\epsilon - \tau$ around $x$.
   Attack the model in eval mode. Then set the model back to train mode.
2. `certified_loss`: Calculate the cross-entropy of $\hat{z} = -\underline{m}$. Here, $\underline{m}$ are the IBP lower
   bounds of the margins on the small ball. Thus $\hat{z}_y = 0$.
   With `reduction="none"`, return the loss of each image.
3. `single_norm_loss`: For $t < 1$, return the standard loss. After that, return `certified_loss` with the radius
   `eps_scale(t) * EPS[norm]`.

The default ratios are `LAM = {"linf": 0.7, "l2": 0.7}`.

In [7]:
LAM = {"linf": 0.7, "l2": 0.7}


def propagation_region(model, x, y, eps, lam, norm):
    # Returns (center, tau).
    tau = lam * eps
    model.eval()
    x_star = pgd(model, x, y, eps - tau, norm, steps=5)
    model.train()
    return x_star, tau


def region_logits(model, x, y, eps, lam, norm):
    c, tau = propagation_region(model, x, y, eps, lam, norm)
    return -ibp_margin_bounds(model, c, tau, norm, y)


def certified_loss(model, x, y, eps, lam, norm, reduction="mean"):
    return F.cross_entropy(region_logits(model, x, y, eps, lam, norm), y, reduction=reduction)


def single_norm_loss(model, x, y, t, norm):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    return certified_loss(model, x, y, eps_scale(t) * EPS[norm], LAM[norm], norm)

In [8]:
models, results = {}, []
for name, loss_fn in [("standard", standard_loss),
                      ("linf", lambda m, x, y, t: single_norm_loss(m, x, y, t, "linf")),
                      ("l2", lambda m, x, y, t: single_norm_loss(m, x, y, t, "l2"))]:
    models[name] = make_model()
    sec = train(models[name], loss_fn)
    results.append(dict(evaluate(models[name], name, attacks=True), time=sec))
print_results(results)

epoch  1  loss 0.5742  time 1 s
epoch  2  loss 0.2150  time 1 s
epoch  3  loss 0.1565  time 1 s
epoch  4  loss 0.1288  time 2 s
epoch  5  loss 0.1000  time 2 s
epoch  6  loss 0.0893  time 2 s
epoch  7  loss 0.0747  time 2 s
epoch  8  loss 0.0644  time 2 s
epoch  9  loss 0.0591  time 3 s
epoch 10  loss 0.0540  time 3 s
epoch 11  loss 0.0459  time 3 s
epoch 12  loss 0.0429  time 3 s
epoch 13  loss 0.0439  time 3 s
epoch 14  loss 0.0348  time 4 s
epoch 15  loss 0.0348  time 4 s
epoch 16  loss 0.0186  time 4 s
epoch 17  loss 0.0142  time 4 s
epoch 18  loss 0.0129  time 5 s
epoch 19  loss 0.0119  time 5 s
epoch 20  loss 0.0113  time 5 s
epoch  1  loss 0.5883  time 0 s
epoch  2  loss 0.4870  time 7 s
epoch  3  loss 0.4765  time 8 s
epoch  4  loss 0.4804  time 10 s
epoch  5  loss 0.4975  time 11 s
epoch  6  loss 0.5255  time 12 s
epoch  7  loss 0.5673  time 14 s
epoch  8  loss 0.6250  time 15 s
epoch  9  loss 0.6905  time 16 s
epoch 10  loss 0.7668  time 17 s
epoch 11  loss 0.8412  time 19 s


# (b) CURE

Let $\hat{z}^\infty$ and $\hat{z}^2$ be the negative IBP margin bounds on the two small balls of an image.
These are the same values as in `certified_loss`. Let $d^p = \mathrm{softmax}(\hat{z}^p)$. Complete `cure_loss`:
$$\mathcal{L}_{CURE} = \frac{1}{n} \sum_{i=1}^{n} \max\left(\mathcal{L}_{CE}(\hat{z}^\infty_i, y_i),\ \mathcal{L}_{CE}(\hat{z}^2_i, y_i)\right)
+ \eta \cdot \frac{1}{|\gamma|} \sum_{i \in \gamma} \mathrm{KL}\left(d^2_i \,\|\, d^\infty_i\right),$$
Here, $\gamma$ is the set of images in the batch that are certified for $\ell_2$ on their small ball.
For these images, all entries of $\hat{z}^2_i$ except $\hat{z}^2_{i, y_i}$ are negative.
The first term is CURE-Max. The second term is the bound alignment (BA).
For $t < 1$, return the standard loss. If $\gamma$ is empty, the second term is 0.

Then complete `gradient_projection` (GP). Let $g_c$ be the gradient of `cure_loss`, and $g_n$ the gradient of the
cross-entropy loss on the clean images. For each parameter tensor:
$$g = \beta \cdot \mathrm{GP}(g_n, g_c) + (1 - \beta) \cdot g_c, \qquad
\mathrm{GP}(g_n, g_c) = \begin{cases} \cos(g_n, g_c) \cdot g_n & \text{if } \cos(g_n, g_c) > 0 \\ 0 & \text{otherwise.} \end{cases}$$
This is the same operation as in RAMP.

This is a simplified version of CURE. The paper is different in three ways:

* It propagates an $\ell_\infty$ box also for $\ell_2$.
* It uses a different $\ell_2$ attack.
* It applies GP to the updates of a full epoch.

In [9]:
def cure_loss(model, x, y, t, eta=0.5):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    z = {p: region_logits(model, x, y, eps_scale(t) * EPS[p], LAM[p], p) for p in NORMS}
    ce = {p: F.cross_entropy(z[p], y, reduction="none") for p in NORMS}
    loss = torch.maximum(ce["linf"], ce["l2"]).mean()
    if eta > 0:
        others = z["l2"].scatter(1, y.unsqueeze(1), -float("inf"))
        gamma = others.max(dim=1).values < 0
        if gamma.any():
            log_d2, log_dinf = F.log_softmax(z["l2"][gamma], 1), F.log_softmax(z["linf"][gamma], 1)
            loss = loss + eta * (log_d2.exp() * (log_d2 - log_dinf)).sum(1).mean()
    return loss


def gradient_projection(g_n, g_c, beta):
    cos = F.cosine_similarity(g_n.flatten(), g_c.flatten(), dim=0)
    gp = torch.where(cos > 0, cos * g_n, torch.zeros_like(g_n))
    return beta * gp + (1 - beta) * g_c

In [10]:
def run(name, loss_fn, seeds=(0, 1), beta=None, epochs=20):
    runs = []
    for s in seeds:
        torch.manual_seed(s)
        model = make_model()
        sec = train(model, loss_fn, epochs=epochs, seed=s, gp_beta=beta, verbose=False)
        runs.append(dict(evaluate(model, name, attacks=True), time=sec))
        print(name, "seed", s, {k: round(v, 2) for k, v in runs[-1].items() if k != "name"})
    mean = {k: (name if k == "name" else float(np.mean([r[k] for r in runs]))) for k in runs[0]}
    return mean, runs, model


b_specs = {"cure_max": (lambda m, x, y, t: cure_loss(m, x, y, t, eta=0.0), None),
           "cure_max_ba": (cure_loss, None),
           "cure_max_ba_gp": (cure_loss, 0.5)}
cure_results = []
for name, (loss_fn, beta) in b_specs.items():
    mean, _, _ = run(name, loss_fn, beta=beta)
    cure_results.append(mean)
print("part (a), seed 0, and part (b), mean of seeds 0 and 1")
print_results(results + cure_results)

cure_max seed 0 {'clean': 92.24, 'linf': 67.47, 'l2': 57.62, 'union': 57.53, 'pgd_linf': 76.72, 'pgd_l2': 77.88, 'pgd_union': 76.06, 'time': 2.47}
cure_max seed 1 {'clean': 92.63, 'linf': 67.75, 'l2': 56.74, 'union': 56.66, 'pgd_linf': 77.75, 'pgd_l2': 78.99, 'pgd_union': 77.12, 'time': 2.46}
cure_max_ba seed 0 {'clean': 91.99, 'linf': 67.35, 'l2': 57.47, 'union': 57.44, 'pgd_linf': 76.26, 'pgd_l2': 78.0, 'pgd_union': 75.74, 'time': 2.6}
cure_max_ba seed 1 {'clean': 92.0, 'linf': 67.08, 'l2': 56.51, 'union': 56.44, 'pgd_linf': 76.51, 'pgd_l2': 77.94, 'pgd_union': 76.01, 'time': 2.61}
cure_max_ba_gp seed 0 {'clean': 92.91, 'linf': 66.96, 'l2': 56.9, 'union': 56.82, 'pgd_linf': 76.53, 'pgd_l2': 78.43, 'pgd_union': 75.95, 'time': 2.79}
cure_max_ba_gp seed 1 {'clean': 92.34, 'linf': 66.93, 'l2': 56.42, 'union': 56.35, 'pgd_linf': 76.37, 'pgd_l2': 77.9, 'pgd_union': 75.88, 'time': 2.75}
part (a), seed 0, and part (b), mean of seeds 0 and 1
standard                  clean  97.29  linf   0.00

# (c) Your method

Write the loss of your method in `my_loss`. Train it with `train` and the default settings, as for the other methods.
If your method needs a different update, write your own copy of `train`.
Do not change the verifier or `evaluate`.

## (c) (i) Idea: train on the same bounds the verifier uses

**Observation.** The verifier proves robustness with the larger of the IBP and CROWN lower bounds, but the CURE loss only sees the IBP bound. For $\ell_2$ the IBP bound is exact in the first layer and then becomes a box, so it is much looser than CROWN. The network is therefore optimized for a bound that is not the one that is verified, and the certified accuracy is limited by bounds the loss never tried to tighten. The cheap check is the gap between the certified and the PGD accuracy for each norm in the table of part (b).

**Method (CURE-Max with CROWN-IBP bounds).** Keep the SABR small ball and the per-image max over the two norms of CURE-Max. Replace the IBP margin bound $\underline{m}^{IBP}$ on the small ball by

$$\underline{m}^p = \max\left(\underline{m}^{IBP},\ \underline{m}^{CROWN\text{-}IBP}\right), \qquad \hat{z}^p = -\underline{m}^p,$$

$$\mathcal{L} = \frac{1}{n}\sum_{i=1}^{n} \max\left(\mathcal{L}_{CE}(\hat{z}^\infty_i, y_i),\ \mathcal{L}_{CE}(\hat{z}^2_i, y_i)\right).$$

$\underline{m}^{CROWN\text{-}IBP}$ is one backward linear bound from the specification, using the IBP bounds of the hidden layers as the ReLU relaxation points. Both terms are valid lower bounds of $f_y - f_j$ on the small ball, so their maximum is valid and at least as tight as each. The extra cost is one backward pass with $10 \times$ width matrices.

**Why it should help.** The loss now measures a bound that is close to the verified one, so the gradient pushes on the quantity that decides certification. The gain should be largest for $\ell_2$, where IBP is loosest.

**Sources and what is new.** CROWN-IBP is from Zhang et al., *Towards Stable and Efficient Training of Verifiably Robust Neural Networks* (ICLR 2020). SABR is from Mueller et al., *Certified Training: Small Boxes are All You Need* (ICLR 2023). CURE is [JCS24] from the homework. The new part is to use the elementwise maximum of IBP and CROWN-IBP on the SABR small ball for both norms of the multi-norm loss, so that the loss uses the same kind of bound as the verifier.

**Pseudocode.**
```
for each batch (x, y) with progress t:
    if t < 1: loss = CE(f(x), y)
    else:
        for p in {linf, l2}:
            eps_p = eps_scale(t) * EPS[p];  tau = LAM[p] * eps_p
            x* = PGD_p(x, y, radius = eps_p - tau, 5 steps)          # eval mode
            m  = max(IBP(x*, tau), CROWN-IBP(x*, tau))                # margins f_y - f_j
            CE_p = cross_entropy(-m, y)                               # per image
        loss = mean_i max(CE_linf_i, CE_l2_i)
```

## (c) (ii) Implementation

In [11]:
def ibp_preactivation_bounds(model, c, r, norm):
    layers = linear_layers(model)
    l, u = first_layer_bounds(layers[0], c, r, norm)
    bounds = [(l, u)]
    for layer in layers[1:-1]:
        l, u = F.relu(l), F.relu(u)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
        l, u = mid - rad, mid + rad
        bounds.append((l, u))
    return bounds


def crown_ibp_margin_bounds(model, c, r, norm, y):
    bounds = ibp_preactivation_bounds(model, c, r, norm)
    return _backward_lower(spec_matrix(y), linear_layers(model), bounds, c, r, norm)


def tight_logits(model, x, y, eps, lam, norm, mode="max"):
    c, tau = propagation_region(model, x, y, eps, lam, norm)
    if mode == "ibp":
        return -ibp_margin_bounds(model, c, tau, norm, y)
    m = crown_ibp_margin_bounds(model, c, tau, norm, y)
    if mode == "max":
        m = torch.maximum(m, ibp_margin_bounds(model, c, tau, norm, y))
    return -m


def make_my_loss(mode="max", tight_norms=("linf", "l2")):
    def loss_fn(model, x, y, t):
        if t < WARMUP:
            return standard_loss(model, x, y, t)
        ce = []
        for p in NORMS:
            z = tight_logits(model, x, y, eps_scale(t) * EPS[p], LAM[p], p, mode if p in tight_norms else "ibp")
            ce.append(F.cross_entropy(z, y, reduction="none"))
        return torch.maximum(ce[0], ce[1]).mean()
    return loss_fn


my_loss = make_my_loss()

## (c) (iii) Results

Two seeds for the new method, the training time for each epoch, and the comparison with the best network of part (b). If the new method is slower than that network, the best network of part (b) is also trained for the same total time (more epochs).

In [12]:
mine, _, my_model = run("cure_max_crown_ibp", my_loss)
best_b = max(cure_results, key=lambda r: r["union"])
print("best of part (b):", best_b["name"])
print_results([best_b, mine])

cure_max_crown_ibp seed 0 {'clean': 94.97, 'linf': 72.93, 'l2': 64.55, 'union': 64.52, 'pgd_linf': 79.8, 'pgd_l2': 80.79, 'pgd_union': 78.94, 'time': 4.15}
cure_max_crown_ibp seed 1 {'clean': 94.29, 'linf': 71.82, 'l2': 63.88, 'union': 63.83, 'pgd_linf': 78.27, 'pgd_l2': 79.07, 'pgd_union': 77.13, 'time': 4.11}
best of part (b): cure_max
cure_max                  clean  92.44  linf  67.61  l2  57.18  union  57.09  pgd_linf  77.23  pgd_l2  78.44  pgd_union  76.59  time/epoch   2.5 s
cure_max_crown_ibp        clean  94.63  linf  72.38  l2  64.22  union  64.17  pgd_linf  79.03  pgd_l2  79.93  pgd_union  78.03  time/epoch   4.1 s


In [13]:
equal_time = []
if mine["time"] > 1.05 * best_b["time"]:
    epochs_eq = round(20 * mine["time"] / best_b["time"])
    loss_fn, beta = b_specs[best_b["name"]]
    eq_mean, _, _ = run(best_b["name"] + f"_{epochs_eq}ep", loss_fn, beta=beta, epochs=epochs_eq)
    equal_time.append(eq_mean)
    print(f"{best_b['name']} trained for {epochs_eq} epochs, about the same total time as the new method")
else:
    print("The new method is not slower than the best network of part (b), so no equal-time run is needed.")
print_results([best_b, mine] + equal_time)

cure_max_33ep seed 0 {'clean': 92.35, 'linf': 67.52, 'l2': 57.63, 'union': 57.58, 'pgd_linf': 77.23, 'pgd_l2': 78.23, 'pgd_union': 76.46, 'time': 2.53}
cure_max_33ep seed 1 {'clean': 92.75, 'linf': 67.59, 'l2': 56.75, 'union': 56.62, 'pgd_linf': 77.47, 'pgd_l2': 78.91, 'pgd_union': 76.73, 'time': 2.53}
cure_max trained for 33 epochs, about the same total time as the new method
cure_max                  clean  92.44  linf  67.61  l2  57.18  union  57.09  pgd_linf  77.23  pgd_l2  78.44  pgd_union  76.59  time/epoch   2.5 s
cure_max_crown_ibp        clean  94.63  linf  72.38  l2  64.22  union  64.17  pgd_linf  79.03  pgd_l2  79.93  pgd_union  78.03  time/epoch   4.1 s
cure_max_33ep             clean  92.55  linf  67.56  l2  57.19  union  57.10  pgd_linf  77.35  pgd_l2  78.57  pgd_union  76.59  time/epoch   2.5 s


Sanity check of the new bound: it must never exceed the true margin at sampled points of the small ball, and it must be at least as tight as IBP.

In [14]:
@torch.no_grad()
def check_bounds(model, n=500, samples=200):
    x, y = x_test[:n], y_test[:n]
    for norm in NORMS:
        c, r = x, 0.5 * EPS[norm]
        m_ibp = ibp_margin_bounds(model, c, r, norm, y)
        m_ci = crown_ibp_margin_bounds(model, c, r, norm, y)
        worst = torch.full_like(m_ci, float("inf"))
        for _ in range(samples):
            d = torch.randn_like(c)
            d = d / d.flatten(1).norm(dim=1).view(-1, 1, 1, 1) * torch.rand(len(c), 1, 1, 1, device=c.device) * r if norm == "l2" else (torch.rand_like(c) * 2 - 1) * r
            z = model((c + d).clamp(0, 1) if norm == "linf" else c + d)
            worst = torch.minimum(worst, z.gather(1, y[:, None]) - z)
        print(f"{norm}: CROWN-IBP above true margin: {(m_ci > worst + 1e-4).sum().item()}, "
              f"IBP above true margin: {(m_ibp > worst + 1e-4).sum().item()}, "
              f"mean CROWN-IBP minus IBP (min over classes): {(m_ci.min(1).values - m_ibp.min(1).values).mean().item():.4f}")

check_bounds(my_model)

linf: CROWN-IBP above true margin: 0, IBP above true margin: 0, mean CROWN-IBP minus IBP (min over classes): 23.2343
l2: CROWN-IBP above true margin: 0, IBP above true margin: 0, mean CROWN-IBP minus IBP (min over classes): 31.4741


## (c) (iv) Ablation

Two changes, one at a time, with seed 0 only:

* `crown_ibp_only`: remove the IBP term of the maximum, so the loss uses the CROWN-IBP bound alone.
* `tight_l2_only`: use the CROWN-IBP maximum only for the $\ell_2$ term and plain IBP for $\ell_\infty$.

The plain IBP loss is `cure_max` from part (b).

In [15]:
ablations = []
for name, loss_fn in [("crown_ibp_only", make_my_loss(mode="crown_ibp")),
                      ("tight_l2_only", make_my_loss(tight_norms=("l2",)))]:
    mean, _, _ = run(name, loss_fn, seeds=(0,))
    ablations.append(mean)
cure_max_row = next(r for r in cure_results if r["name"] == "cure_max")
print_results([cure_max_row, mine] + ablations)

crown_ibp_only seed 0 {'clean': 94.87, 'linf': 72.69, 'l2': 64.51, 'union': 64.48, 'pgd_linf': 79.51, 'pgd_l2': 80.47, 'pgd_union': 78.51, 'time': 3.27}
tight_l2_only seed 0 {'clean': 92.61, 'linf': 71.61, 'l2': 56.89, 'union': 56.88, 'pgd_linf': 77.07, 'pgd_l2': 76.43, 'pgd_union': 74.95, 'time': 3.29}
cure_max                  clean  92.44  linf  67.61  l2  57.18  union  57.09  pgd_linf  77.23  pgd_l2  78.44  pgd_union  76.59  time/epoch   2.5 s
cure_max_crown_ibp        clean  94.63  linf  72.38  l2  64.22  union  64.17  pgd_linf  79.03  pgd_l2  79.93  pgd_union  78.03  time/epoch   4.1 s
crown_ibp_only            clean  94.87  linf  72.69  l2  64.51  union  64.48  pgd_linf  79.51  pgd_l2  80.47  pgd_union  78.51  time/epoch   3.3 s
tight_l2_only             clean  92.61  linf  71.61  l2  56.89  union  56.88  pgd_linf  77.07  pgd_l2  76.43  pgd_union  74.95  time/epoch   3.3 s


## Final table

All rows use `evaluate`. Rows from part (b) and (c) are the mean of two seeds, the ablations use seed 0, and the single-norm rows of part (a) use seed 0.

In [16]:
import json
final = results + cure_results + [mine] + equal_time + ablations
print_results(final)
json.dump(final, open("results_p2.json", "w"), indent=1)

standard                  clean  97.29  linf   0.00  l2   0.00  union   0.00  pgd_linf   0.53  pgd_l2   9.72  pgd_union   0.51  time/epoch   0.3 s
linf                      clean  93.52  linf  80.88  l2   5.51  union   5.51  pgd_linf  84.63  pgd_l2  71.43  pgd_union  71.20  time/epoch   1.6 s
l2                        clean  93.14  linf  54.57  l2  58.23  union  51.70  pgd_linf  68.58  pgd_l2  78.14  pgd_union  68.14  time/epoch   1.3 s
cure_max                  clean  92.44  linf  67.61  l2  57.18  union  57.09  pgd_linf  77.23  pgd_l2  78.44  pgd_union  76.59  time/epoch   2.5 s
cure_max_ba               clean  92.00  linf  67.22  l2  56.99  union  56.94  pgd_linf  76.39  pgd_l2  77.97  pgd_union  75.88  time/epoch   2.6 s
cure_max_ba_gp            clean  92.62  linf  66.94  l2  56.66  union  56.59  pgd_linf  76.45  pgd_l2  78.17  pgd_union  75.91  time/epoch   2.8 s
cure_max_crown_ibp        clean  94.63  linf  72.38  l2  64.22  union  64.17  pgd_linf  79.03  pgd_l2  79.93  pgd_unio

## (c) (v) Analysis

The method works. Union certified accuracy rises from 57.09% (CURE-Max) to 64.17%, and clean accuracy from 92.44% to 94.63%. Both seeds of the new method (64.52%, 63.83%) beat both seeds of CURE-Max (57.53%, 56.66%).

- The gap between certified and PGD accuracy shrinks: l_inf 9.6 to 6.7 points, l2 21.3 to 15.7 points, union 19.5 to 13.9 points. PGD union accuracy rises only from 76.59% to 78.03%, so most of the gain is tighter certification.
- CURE-Max trained for 33 epochs (same total time) gives 57.10% union, so extra training does not explain the gain.
- CROWN-IBP only gives 64.48% union (seed 0), the same as the full method on that seed (64.52%) and faster. The IBP term of the maximum adds nothing here.
- Using the tight bound only for l2 gives 56.88% union, with no l2 gain. A likely cause is the per-image maximum: the looser l_inf IBP loss is selected most of the time, so the l2 term gets few gradients. I did not measure how often each term is selected.
- l2 still limits the union (64.22% against 72.38% for l_inf), and a 13.9 point gap to PGD remains. The loss uses CROWN-IBP, which is looser than the full CROWN bound of the verifier.
- Limits: two seeds, one seed for the ablations, a 20-epoch schedule.
